# Combined preprocessing, feature extraction, and segmentation

Set `image_limit` below to control number of images. The notebook will sample images from different class folders rather than from the same folder. The flow is: preprocessing -> segmentation -> features.

## Setup

Define dataset locations and runtime parameters.

In [1]:
import os
import sys
import random
from pathlib import Path
import importlib.util
import types
import json

# Notebook parameters
image_limit = 20  # number of images to process (change as needed)
input_root = Path('..') / 'data' / 'processed' / 'PlantVillage'
if not input_root.exists():
    input_root = Path('..') / 'data' / 'processed'
input_root = input_root.resolve()
output_root = Path('..') / 'results' / 'combined_run'
output_root.mkdir(parents=True, exist_ok=True)
size = (224, 224)
color_mode = 'rgb'  # 'rgb','hsv','gray'
denoise = 'none'  # 'none','gaussian','median','bilateral'
random_seed = 42
random.seed(random_seed)
print('input_root:', input_root)
print('output_root:', output_root)
print('image_limit:', image_limit)

# Prepare a package-like `src` entry and load modules by file location
repo_root = Path('..').resolve()
src_dir = repo_root / 'src'
src_pkg = types.ModuleType('src')
src_pkg.__path__ = [str(src_dir.resolve())]
sys.modules['src'] = src_pkg

def _load(name, path):
    spec = importlib.util.spec_from_file_location(name, str(path))
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    spec.loader.exec_module(module)
    return module

features_mod = _load('src.features', src_dir / 'features.py')
segmentation_mod = _load('src.segmentation', src_dir / 'segmentation.py')
preprocessing_mod = _load('src.preprocessing', src_dir / 'preprocessing.py')
print('Loaded modules:', features_mod.__name__, segmentation_mod.__name__, preprocessing_mod.__name__)


input_root: C:\Users\louay\OneDrive\Desktop\computer vision\data\processed\PlantVillage
output_root: ..\results\combined_run
image_limit: 20
Loaded modules: src.features src.segmentation src.preprocessing


## Image sampling utilities

Functions to find image-containing directories and to sample images across different folders.

In [2]:
import os
import random
from pathlib import Path
IMAGE_EXTS = ('.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff', '.webp')

def find_image_dirs(root):
    image_dirs = []
    for dirpath, dirnames, filenames in os.walk(str(root)):
        for f in filenames:
            if f.lower().endswith(IMAGE_EXTS):
                image_dirs.append(dirpath)
                break
    return image_dirs

def sample_images_from_unique_dirs(root, limit):
    dirs = find_image_dirs(root)
    dirs = sorted(set(dirs))
    if not dirs:
        raise FileNotFoundError(f'No image directories found under {root}')
    random.shuffle(dirs)
    selected = []
    # pick at most one image per directory to maximize folder diversity
    for d in dirs:
        if len(selected) >= limit:
            break
        files = [os.path.join(d, f) for f in os.listdir(d) if f.lower().endswith(IMAGE_EXTS)]
        if files:
            selected.append(os.path.abspath(random.choice(files)))
    # if we still need more images, sample additional images across all directories
    if len(selected) < limit:
        all_files = []
        for d in dirs:
            all_files.extend([os.path.join(d, f) for f in os.listdir(d) if f.lower().endswith(IMAGE_EXTS)])
        remaining = limit - len(selected)
        candidates = list(set(all_files) - set(selected))
        if len(candidates) >= remaining:
            selected.extend(random.sample(candidates, remaining))
        else:
            if candidates:
                selected.extend(candidates)
                still = remaining - len(candidates)
            else:
                still = remaining
            if still > 0 and all_files:
                selected.extend(random.choices(all_files, k=still))
    return [Path(p) for p in selected]

## Select images

Sample `image_limit` images ensuring folder diversity and print a short preview.

In [3]:
selected_images = sample_images_from_unique_dirs(input_root, image_limit)
print(f'Selected {len(selected_images)} images from {len(set(p.parent for p in selected_images))} folders')
for p in selected_images[:min(30, len(selected_images))]:
    print(p)

Selected 20 images from 15 folders
C:\Users\louay\OneDrive\Desktop\computer vision\data\processed\PlantVillage\Tomato_Leaf_Mold\0a555f63-bf03-4958-8993-e1932b8dce9f___Crnl_L.Mold 9064.JPG
C:\Users\louay\OneDrive\Desktop\computer vision\data\processed\PlantVillage\Tomato__Tomato_mosaic_virus\0befa341-0db3-49f4-b4fc-beeb05854bff___PSU_CG 2338.JPG
C:\Users\louay\OneDrive\Desktop\computer vision\data\processed\PlantVillage\Tomato_Late_blight\1627cb48-dc80-46a1-bda0-b677c32fbf56___RS_Late.B 5066.JPG
C:\Users\louay\OneDrive\Desktop\computer vision\data\processed\PlantVillage\Tomato_Early_blight\342d78c9-af97-4325-ad0f-0f12e0a206c2___RS_Erly.B 7406.JPG
C:\Users\louay\OneDrive\Desktop\computer vision\data\processed\PlantVillage\Tomato_healthy\4a1e2b71-992a-4a64-a599-b49b8fa75378___RS_HL 0627.JPG
C:\Users\louay\OneDrive\Desktop\computer vision\data\processed\PlantVillage\Tomato__Tomato_YellowLeaf__Curl_Virus\6262e790-cfd9-4e45-bbbc-77db5c2064e7___YLCV_NREC 2328.JPG
C:\Users\louay\OneDrive\Deskt

## Preprocessing

Resize, denoise and save processed images (keeps original folder-relative structure under `results/combined_run/processed`).

In [4]:
import cv2
processed_dir = output_root / 'processed'
processed_dir.mkdir(parents=True, exist_ok=True)

processed_items = []
for p in selected_images:
    try:
        rel = p.relative_to(input_root)
    except Exception:
        rel = Path(p).name
    out_p = processed_dir / rel
    out_p.parent.mkdir(parents=True, exist_ok=True)
    img_bgr = cv2.imread(str(p))
    if img_bgr is None:
        print(f'Failed to read {p}, skipping')
        continue
    img_bgr = cv2.resize(img_bgr, (size[0], size[1]), interpolation=cv2.INTER_AREA)
    img_bgr = preprocessing_mod.apply_denoise(img_bgr, denoise)
    # produce color versions for downstream steps
    img_rgb = preprocessing_mod.convert_color(img_bgr, 'rgb')
    img_hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    img_gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    # save processed image (saved according to `color_mode` param)
    preprocessing_mod.save_output_image(img_rgb if color_mode == 'rgb' else (img_hsv if color_mode == 'hsv' else img_gray), out_p, color_mode)
    processed_items.append({'orig_path': str(p), 'processed_path': str(out_p)})

print('Preprocessing done. Processed:', len(processed_items))

Preprocessing done. Processed: 20


## Segmentation

Run the segmentation pipeline on each processed image and save masks + refined leaf crops.

In [5]:
import cv2
masks_dir = output_root / 'masks'
masks_dir.mkdir(parents=True, exist_ok=True)
leaf_dir = output_root / 'leaf'
leaf_dir.mkdir(parents=True, exist_ok=True)

segmentation_results = []
for item in processed_items:
    proc_path = Path(item['processed_path'])
    img_bgr = cv2.imread(str(proc_path))
    if img_bgr is None:
        img_bgr = cv2.imread(item['orig_path'])
        if img_bgr is None:
            print(f"Cannot read {proc_path} or original {item['orig_path']}, skipping")
            continue
    img_bgr = cv2.resize(img_bgr, (size[0], size[1]), interpolation=cv2.INTER_AREA)
    image_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    image_gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    image_hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)

    seg = segmentation_mod.segment_leaf(image_rgb, image_gray, image_hsv, n_clusters=3, kernel_size=5, grabcut_iter=5, seed=random_seed)
    mask_refined = seg.get('refined')
    base_name = proc_path.stem
    mask_refined_path = masks_dir / f'{base_name}_mask_refined.png'
    if mask_refined is not None:
        cv2.imwrite(str(mask_refined_path), mask_refined)
    leaf_refined = seg.get('leaf_refined')
    if leaf_refined is not None:
        cv2.imwrite(str(leaf_dir / f'{base_name}_leaf_refined.png'), cv2.cvtColor(leaf_refined, cv2.COLOR_RGB2BGR))
    stats = segmentation_mod.mask_stats(mask_refined)
    segmentation_results.append({'orig_path': item['orig_path'], 'processed_path': str(proc_path), 'mask_refined': str(mask_refined_path), 'stats': stats})

print('Segmentation complete for', len(segmentation_results), 'images')

Segmentation complete for 20 images


## Feature extraction

Compute features for each segmented image (uses `src.features.extract_features`) and save JSON files under `results/combined_run/features`.

In [6]:
features_dir = output_root / 'features'
features_dir.mkdir(parents=True, exist_ok=True)
feature_results = []
for seg_item in segmentation_results:
    proc_path = Path(seg_item['processed_path'])
    img_bgr = cv2.imread(str(proc_path))
    if img_bgr is None:
        img_bgr = cv2.imread(seg_item['orig_path'])
        if img_bgr is None:
            print(f'Cannot read {proc_path}, skipping feature extraction')
            continue
    image_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    mask = cv2.imread(seg_item['mask_refined'], cv2.IMREAD_GRAYSCALE)
    if mask is None:
        mask = None
    else:
        mask = (mask > 0).astype('uint8')
    vector, details = features_mod.extract_features(image_rgb, mask=mask)
    feat_path = features_dir / f"{Path(seg_item['orig_path']).stem}_features.json"
    details_serializable = {
        'rgb_hist': details['rgb_hist'].tolist() if hasattr(details['rgb_hist'], 'tolist') else details['rgb_hist'],
        'hsv_hist': details['hsv_hist'].tolist() if hasattr(details['hsv_hist'], 'tolist') else details['hsv_hist'],
        'glcm': details['glcm'],
        'shape': details['shape'],
    }
    feat_json = {'orig_path': seg_item['orig_path'], 'processed_path': seg_item['processed_path'], 'mask': seg_item['mask_refined'], 'vector': vector.tolist(), 'details': details_serializable}
    feat_path.write_text(json.dumps(feat_json, indent=2), encoding='utf-8')
    feature_results.append({'orig_path': seg_item['orig_path'], 'features_path': str(feat_path)})

print('Features computed for', len(feature_results), 'images')

Features computed for 20 images


## Save combined results

Write a summary JSON with selected, processed, segmentation and feature file paths.

In [7]:
combined = {
    'selected_images': [str(p) for p in selected_images],
    'processed': [{'orig_path': x['orig_path'], 'processed_path': x['processed_path']} for x in processed_items],
    'segmentation': segmentation_results,
    'features': feature_results,
}
combined_path = output_root / 'combined_results.json'
combined_path.write_text(json.dumps(combined, indent=2), encoding='utf-8')
print('Saved combined results to', combined_path)

Saved combined results to ..\results\combined_run\combined_results.json
